# 12 — Agent Teams

Dynamic, LLM-routed multi-agent collaboration. Define agents with roles, and a coordinator decides who works, in what order, when to loop back. No manual graph wiring needed.

**What you'll learn:**
1. Create team agents with names and roles
2. Set up a coordinator LLM
3. Run a team on a complex task
4. Inspect round-by-round delegation history
5. Control max rounds for safety

In [1]:
from pathlib import Path
import sys

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent import Agent, AgentTeam, TeamAgent
from examples.run_multi_tool_agent import build_llm_from_env
from IPython.display import display, Markdown

llm = build_llm_from_env("bedrock")

## 1. Define Team Members

Each `TeamAgent` has a name, role description, and an actual `Agent` instance that does the work.

In [2]:
# Define team members with specialized roles
researcher = TeamAgent(
    name="researcher",
    role="Expert at researching topics and gathering key facts. Returns concise bullet points.",
    agent=Agent(
        llm=llm,
        prompt="You are a research expert. Provide key facts as concise bullet points.",
    ),
    capabilities=["research", "fact-finding", "analysis"],
)

writer = TeamAgent(
    name="writer",
    role="Expert at writing clear, engaging content from research notes. Creates polished prose.",
    agent=Agent(
        llm=llm,
        prompt="You are a skilled technical writer. Write clear, engaging content.",
    ),
    capabilities=["writing", "editing", "content creation"],
)

reviewer = TeamAgent(
    name="reviewer",
    role="Critical reviewer who checks content for accuracy, clarity, and completeness. Can send work back for revision.",
    agent=Agent(
        llm=llm,
        prompt="You are a critical reviewer. Point out issues concisely. Say APPROVED if the content is ready.",
    ),
    capabilities=["review", "quality assurance", "feedback"],
)

print(f"Team members: {researcher.name}, {writer.name}, {reviewer.name}")

Team members: researcher, writer, reviewer


## 2. Create and Run the Team

The coordinator LLM sees agent descriptions and decides who should work on what. It routes dynamically based on the conversation.

In [3]:
# Create team with coordinator
team = AgentTeam(
    name="content-team",
    coordinator=llm,  # LLM that routes work between agents
    agents=[researcher, writer, reviewer],  # available team members
    max_rounds=6,  # safety limit
)

# Run the team on a task
result = team.run(
    "Write a concise technical overview of WebAssembly (WASM) and why it matters."
)

# Show round-by-round delegation history
print("=== Team Delegation Log ===")
for r in result.rounds:
    print(f"\nRound {r.number} [{r.agent}]")
    print(f"  Task: {r.prompt[:100]}...")
    print(f"  Output: {r.output[:150]}...")

print(f"\n=== Final Output ({len(result.rounds)} rounds) ===")
display(Markdown(result.output))

=== Team Delegation Log ===

Round 1 [researcher]
  Task: Provide concise bullet-point facts for a technical overview of WebAssembly (WASM). Include: what it ...
  Output: **WebAssembly (WASM) – Technical Overview**

---

### What It Is
- **Binary instruction format** for a **portable, low‑level virtual machine** that ru...

Round 2 [researcher]
  Task: Please expand the technical overview of WebAssembly (WASM) into a complete set of concise bullet poi...
  Output: **WebAssembly (WASM) – Technical Overview (concise bullet format)**  

---

### 1. Core Architecture & Execution Model  
- **Stack‑machine ISA** – ope...

Round 3 [researcher]
  Task: Please finish the concise technical overview of WebAssembly (WASM). Include brief bullet points on:
...
  Output: - **Near‑native performance** – binary format enables direct translation to machine code; runtimes can reach ≈ 95 % of native speeds for compute‑heavy...

Round 4 [writer]
  Task: Using the bullet points provided by the researcher i

**WebAssembly (WASM) – Technical Overview**

1. **Definition & Purpose**  
WebAssembly is a low‑level, binary instruction format designed as a portable compilation target for high‑level languages (C/C++, Rust, Go, etc.). It enables developers to run compute‑intensive code at near‑native speed inside web browsers and other runtimes while preserving the safety guarantees of the host environment.

2. **Core Architecture & Execution Model**  
WASM defines a stack‑machine ISA where each instruction pushes or pops typed values on an explicit operand stack. Code is organized into *modules* that encapsulate functions, tables, memories, globals, and import/export descriptors. Two textual representations exist: the compact binary format (≈10 × smaller than JavaScript) for efficient transport, and the human‑readable .wat format for debugging. Compilation can be **JIT** (just‑in‑time) in browsers, **AOT** (ahead‑of‑time) for native binaries, or **Streaming** where decoding and compilation overlap with network transfer, minimizing start‑up latency.

3. **Performance Characteristics**  
WASM delivers performance within 5‑15 % of native C/C++ code because its instruction set maps cleanly to modern CPU pipelines and supports SIMD, multi‑threading, and bulk memory operations. Deterministic execution stems from a well‑specified abstract machine—no hidden side‑effects, explicit memory bounds, and strict type checking—making performance predictable across platforms.

4. **Security & Sandboxing**  
Modules execute in a *linear memory* sandbox isolated from the host’s address space. All accesses are bounds‑checked, and the only interaction with the outside world occurs through explicitly imported functions (e.g., DOM APIs). This design eliminates buffer‑overflows, code injection, and arbitrary system calls, allowing untrusted code to run safely alongside trusted JavaScript.

5. **Portability & Runtime Ecosystem**  
Every major browser implements a WASM engine (V8, SpiderMonkey, WebKit). Outside the browser, runtimes such as Wasmtime, Wasmer, and Wasm3 bring WASM to servers, edge platforms, and embedded devices, often with small footprints (≈100 KB). The ecosystem includes toolchains (LLVM, Emscripten, Rustc) and standards for extensions like threads, SIMD, and reference types.

6. **Why It Matters**  
WASM democratizes high‑performance computing on the web, letting games, AI inference, scientific simulations, and serverless functions run everywhere without recompilation. Its language‑agnostic model breaks the JavaScript monopoly, enabling seamless interoperability and opening new domains such as low‑latency IoT edge analytics and cross‑platform desktop applications.

*In essence, WebAssembly provides a fast, secure, and universally portable execution layer that extends the reach of compiled code from browsers to the cloud and the edge.*

In [ ]:
# Create team with coordinator
team = AgentTeam(
    name="content-team",
    coordinator=llm,  # LLM that routes work between agents
    agents=[researcher, writer, reviewer],  # available team members
    max_rounds=6,  # safety limit
)

# Run the team on a task
# result = team.run("Write a concise technical overview of WebAssembly (WASM) and why it matters.")


for event in team.stream(
    "Write a concise technical overview of WebAssembly (WASM) and why it matters."
):
    print(f"\n=== Event: {event.type} ===")
    print(f"Agent: {event}")


#


=== Event: run_started ===
Agent: AgentEvent(type='run_started', message="Team 'content-team' started", payload={'task': 'Write a concise technical overview of WebAssembly (WASM) and why it matters.', 'agents': ['researcher', 'writer', 'reviewer']})

=== Event: planning_started ===
Agent: AgentEvent(type='planning_started', message='Round 1: Coordinator deciding next step', payload={'round': 1})

=== Event: tool_called ===
Agent: AgentEvent(type='tool_called', message='Delegating to researcher: Provide concise bullet-point facts for a technical overview of WebAssembly (WASM', payload={'agent': 'researcher', 'prompt': 'Provide concise bullet-point facts for a technical overview of WebAssembly (WASM). Include: definition, core design goals, binary format, execution model, supported languages, performance characteristics, security model, browser support, ecosystem (tooling, runtimes, standards), and key reasons why it matters (e.g., performance, portability, cross‑platform code reuse, be